In [ ]:
import pandas as pd
import numpy as np
#pd.set_option('display.max_columns', None)
#pd.set_option('display.max_rows', None)
#np.set_printoptions(threshold=np.inf, linewidth=np.inf)

import matplotlib.pyplot as plt
import seaborn as sns
import sidetable as stb
from datetime import datetime
import folium
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = [10,6]
%matplotlib inline

In [ ]:
original_ais = pd.read_csv("1years_ais.csv")
original_ais

## - 기초통계 확인

In [ ]:
length_data = original_ais[(original_ais["beam"]!=0) & (original_ais["length"]!=0)]
length_data

In [ ]:
length_data[length_data['length']==9]

In [ ]:
length_data['draught'].describe()

In [ ]:
plt.boxplot(length_data['beam'])
plt.show()

In [ ]:
q3 = length_data['length'].quantile(0.75)
q1 = length_data['length'].quantile(0.25)
iqr = q3 - q1

In [ ]:
condition = (length_data['length'] > q3 + 1.5 * iqr) | (length_data['length'] < q1 - 1.5 * iqr)

In [ ]:
a=length_data[condition].index

In [ ]:
length_data.drop(a,inplace=True)

In [ ]:
length_data

## - shiptype, MMSI등 빈도 분석

In [ ]:
original_ais

In [ ]:
#destination, 
original_ais[original_ais['destination'].isnull()]

In [ ]:
original_ais['rot'].describe()

In [ ]:
b = original_ais["shiptype"].value_counts()
b

In [ ]:
original_ais.stb.freq(["navi"])

## unixtime → datetime

In [ ]:
for i in range(len(original_ais)):
    original_ais["utctime"][i] = datetime.fromtimestamp(int(original_ais["utctime"][i]))

In [ ]:
for i in range(len(original_ais)):
    original_ais["eta"][i] = datetime.fromtimestamp(int(original_ais["eta"][i]))

In [ ]:
original_ais["eta"]

In [ ]:
original_ais["utctime"]

In [ ]:
original_ais['ais_cdatetime']

In [ ]:
original_ais.to_csv('preprocess_ais.csv')

In [ ]:
datetime.fromtimestamp(1597130099)

In [ ]:
original_ais.set_index("eta(unixTime)",inplace=True)

In [ ]:
original_ais = original_ais.sort_index(ascending=True)
original_ais

In [ ]:
original_ais.index[0]

## 지도 시각화

In [ ]:
sample = original_ais.sample(n=100)

In [ ]:
sample = sample[['lat','lon']]
sample

In [ ]:
lat, lon = original_ais['lat'][1], original_ais['lon'][1]

In [ ]:
m = folium.Map(location=[lat,lon],zoom_start=5)

for _, row in sample.iterrows():
    lat, lon = row['lat'], row['lon']
    folium.Marker(location=[lat, lon]).add_to(m)

In [ ]:
m

## 상관계수 확인

In [ ]:
heatmap_data = original_ais[['length','beam','draught','hdg','sog','cog','rot']]

In [ ]:
colormap = plt.cm.PuBu
plt.figure(figsize=(10, 8))
sns.heatmap(heatmap_data.astype(float).corr(), linewidths = 0.1, vmax = 1.0,
           square = True, cmap = colormap, linecolor = "white", annot = True, annot_kws = {"size" : 16})